# Retail Footfall Analytics Using YOLO 

## Background

In retail, customer footfall is important for **store planning, staffing, and sales analysis**.

Traditional methods like:

* Manual counting
* Infrared sensors
* Entry systems

are often **costly, inaccurate, and not scalable**.

With computer vision, models like **YOLO** can detect people in real time. However, detection alone leads to **overcounting**, as the same person appears in multiple frames.

To solve this, **object tracking** is used to assign a **unique ID** to each person, enabling accurate footfall counting.

**To address this challenge, we use YOLOv8 combined with BoT-SORT tracking.

- **YOLO (You Only Look Once)** performs real-time person detection in each video frame.
- **BoT-SORT (Better Online Tracking)** assigns a unique and consistent ID to each detected person across frames.
## Objective 

Develop a video-based footfall counting system that:

1. Detects people using a pretrained YOLO model
2. Tracks individuals across frames
3. Assigns unique IDs

### Import Libraries

In [1]:
from ultralytics import YOLO
import cv2

### Load YOLO Model


In [2]:
model = YOLO('yolov8n.pt')



### Load Video


In [3]:
video_path = "Footfall video.mp4"
cap = cv2.VideoCapture(video_path)

# Get video properties
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
fps = int(cap.get(cv2.CAP_PROP_FPS))

print(f"Resolution: {frame_width}x{frame_height}, FPS: {fps}")

# =========================
# Output Video Writer
# =========================
out = cv2.VideoWriter(
    "output_tracking.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (frame_width, frame_height)
)


Resolution: 1080x1920, FPS: 23


### Settings

In [4]:
PERSON_CLASS_ID = 0

# Unique counting
counted_ids = set()

# Line for counting (middle of frame)
line_y = frame_height // 2

# Window settings (fix zoom issue)
cv2.namedWindow("Footfall Tracking", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Footfall Tracking", 960, 540)

###  Process Video Frame-by-Frame

**Note :**
Counting occurs per processed frame, and since processing time varies, the interval is non-uniform and depends on model speed.

In [5]:
# =========================
# Process Video Frame-by-Frame
# =========================
while cap.isOpened():
    # Read each frame from video
    ret, frame = cap.read()

    # Break loop if video ends
    if not ret:
        break

    # =========================
    # Object Detection + Tracking (BoT-SORT)
    # =========================
    results = model.track(
        frame,                 # Input frame
        persist=True,          # Keep track IDs across frames
        tracker="botsort.yaml",# Use BoT-SORT for stable tracking
        conf=0.4,              # Confidence threshold
        iou=0.5,               # IOU threshold for detection
        verbose=False          # Disable logs
    )

    # Extract detected bounding boxes
    boxes = results[0].boxes

    # =========================
    # Footfall Counting Logic (Line Crossing)
    # =========================
    if boxes.id is not None:  # Ensure tracking IDs exist
        for box, track_id in zip(boxes, boxes.id):

            # Filter only 'person' class (class 0 in COCO)
            if int(box.cls) == PERSON_CLASS_ID:
                track_id = int(track_id)

                # Extract bounding box coordinates
                x1, y1, x2, y2 = map(int, box.xyxy[0])

                # Compute center of bounding box (vertical position)
                center_y = (y1 + y2) // 2

                # Count person only once when crossing the line
                if center_y > line_y and track_id not in counted_ids:
                    counted_ids.add(track_id)

    # Total unique people counted
    footfall = len(counted_ids)

    # =========================
    # Visualization
    # =========================
    # Draw bounding boxes + IDs on frame
    annotated_frame = results[0].plot()

    # Draw horizontal counting line
    cv2.line(
        annotated_frame,
        (0, line_y),                 # Start point
        (frame_width, line_y),       # End point
        (0, 0, 255),                # Red color
        2                           # Thickness
    )

    # Display footfall count on screen
    cv2.putText(
        annotated_frame,
        f"Footfall Count: {footfall}",
        (20, 50),                   # Position
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,                        # Font size
        (0, 255, 0),                # Green color
        3,
        cv2.LINE_AA
    )

    # =========================
    # Resize Frame for Display (Fix Zoom Issue)
    # =========================
    display_frame = cv2.resize(
        annotated_frame,
        (960, 820)  # Custom display size (may distort aspect ratio)
    )

    # Show output window
    cv2.imshow("Footfall Tracking", display_frame)

    # =========================
    # Save Output Video
    # =========================
    out.write(annotated_frame)  # Save original resolution frame

    # =========================
    # Exit Condition
    # =========================
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

**Explanation for footfall counting (line crossing logic)**

Without using line-crossing logic, the system counts detections frame by frame, which leads to the same person being counted multiple times as they appear in consecutive frames. This results in significant overcounting and inaccurate footfall estimation.

To overcome this, a virtual line is introduced, and a person is counted only when they cross this line. Combined with tracking IDs, this ensures that each individual is counted only once, improving the accuracy and reliability of the system.


### Release

In [6]:
cap.release()
out.release()
cv2.destroyAllWindows()

print("✅ Output video saved as output_tracking.mp4")
print(f"👥 Total Footfall Count: {footfall}")

✅ Output video saved as output_tracking.mp4
👥 Total Footfall Count: 28


### Limitation
Although the model assigns unique tracking IDs to individuals, there are limitations in accurately estimating the true number of unique people:

- ID Switching: In crowded scenes or occlusions, the tracker may assign a new ID to the same person, leading to overcounting.

- Re-entry Issue: If a person leaves the frame and re-enters, they may be assigned a new ID, and counted again.

- Tracking Loss: Temporary loss of detection (e.g., fast movement, poor lighting) can break tracking continuity.

- Camera Dependency: Accuracy depends heavily on camera angle, resolution, and crowd density.